# Titanic Dataset - Data Preprocessing

**Objective:** Perform complete data preprocessing on the Titanic dataset
and prepare it for machine learning modeling.

## 1. Load & Explore Data

In [1]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
# Import dataset using Pandas
df = pd.read_csv("titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
print("Shape of dataset:", df.shape)

Shape of dataset: (891, 12)


In [4]:
print("Columns:", list(df.columns))

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [5]:
print("Data types:\n")
print(df.dtypes)

Data types:

PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


In [6]:
print("Statistical summary:\n")
df.describe(include="all")

Statistical summary:



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


In [7]:
print("Missing values per column:\n")
print(df.isnull().sum())

Missing values per column:

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [8]:
print("Number of duplicate rows:", df.duplicated().sum())

Number of duplicate rows: 0


In [9]:
print("Target distribution (Survived):\n")
print(df["Survived"].value_counts())
print("\nPercentage:\n")
print(df["Survived"].value_counts(normalize=True) * 100)

Target distribution (Survived):

Survived
0    549
1    342
Name: count, dtype: int64

Percentage:

Survived
0    61.616162
1    38.383838
Name: proportion, dtype: float64


## 2. Data Cleaning

In [10]:
# Remove duplicate records
before = df.shape[0]
df = df.drop_duplicates()
after = df.shape[0]
print(f"Duplicates removed: {before - after}")

Duplicates removed: 0


In [11]:
# Handle missing values
# Age -> Median
df["Age"] = df["Age"].fillna(df["Age"].median())

# Fare -> Median
df["Fare"] = df["Fare"].fillna(df["Fare"].median())

# Embarked -> Mode
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("Missing values after handling Age, Fare, Embarked:")
print(df[["Age", "Fare", "Embarked"]].isnull().sum())

Missing values after handling Age, Fare, Embarked:
Age         0
Fare        0
Embarked    0
dtype: int64


In [12]:
# Analyze the Cabin column
print("Missing values in Cabin:", df["Cabin"].isnull().sum(),
      "out of", df.shape[0], "rows")
print("Percentage missing:", round(df["Cabin"].isnull().mean() * 100, 2), "%")

# Cabin has too many missing values (~77%) to reliably impute.
# Instead of dropping it completely, we convert it into a useful flag:
# HasCabin = 1 if cabin info is known, 0 if missing.
df["HasCabin"] = df["Cabin"].notnull().astype(int)
df = df.drop(columns=["Cabin"])

print("\nNew 'HasCabin' feature created, original 'Cabin' column dropped.")

Missing values in Cabin: 687 out of 891 rows
Percentage missing: 77.1 %

New 'HasCabin' feature created, original 'Cabin' column dropped.


In [13]:
# Remove irrelevant features that don't help prediction
# (PassengerId is just a row id, Ticket is a messy free-text id,
# Name is used below to extract Title and then dropped)
df["Title"] = df["Name"]  # keep a copy before dropping, used in feature engineering step

df = df.drop(columns=["PassengerId", "Ticket"])
print("Remaining columns:", list(df.columns))

Remaining columns: ['Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'HasCabin', 'Title']


## 3. Feature Engineering

In [14]:
# Family Size = SibSp + Parch + 1 (the +1 counts the passenger themself)
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

In [15]:
# IsAlone feature
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

In [16]:
# Extract Passenger Title from Name (Mr, Mrs, Miss, etc.)
df["Title"] = df["Title"].str.extract(r",\s*([^\.]*)\.")
df["Title"] = df["Title"].str.strip()

# Group rare titles into broader categories
title_map = {
    "Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
    "Lady": "Rare", "Countess": "Rare", "the Countess": "Rare", "Capt": "Rare", "Col": "Rare",
    "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare",
    "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare",
}
df["Title"] = df["Title"].replace(title_map)

print("Title value counts:\n")
print(df["Title"].value_counts())

Title value counts:

Title
Mr        517
Miss      185
Mrs       126
Master     40
Rare       23
Name: count, dtype: int64


In [17]:
# Name is no longer needed after Title extraction
df = df.drop(columns=["Name"])
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,HasCabin,Title,FamilySize,IsAlone
0,0,3,male,22.0,1,0,7.2500,S,0,Mr,2,0
1,1,1,female,38.0,1,0,71.2833,C,1,Mrs,2,0
2,1,3,female,26.0,0,0,7.9250,S,0,Miss,1,1
3,1,1,female,35.0,1,0,53.1000,S,1,Mrs,2,0
4,0,3,male,35.0,0,0,8.0500,S,0,Mr,1,1


## 4. Data Transformation

In [18]:
# Encode categorical variables (Sex, Embarked, Title)
# One-hot encoding is used so no artificial order is implied.
df_encoded = pd.get_dummies(df, columns=["Sex", "Embarked", "Title"], drop_first=True)
print("Columns after encoding:", list(df_encoded.columns))

Columns after encoding: ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'FamilySize', 'IsAlone', 'Sex_male', 'Embarked_Q', 'Embarked_S', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']


In [19]:
# Scale numerical features (Age, Fare, FamilySize)
scaler = StandardScaler()
num_cols = ["Age", "Fare", "FamilySize"]
df_encoded[num_cols] = scaler.fit_transform(df_encoded[num_cols])
df_encoded[num_cols].describe()

,Age,Fare,FamilySize
count,8.910000e+02,8.910000e+02,8.910000e+02
mean,2.272780e-16,3.987333e-18,-2.392400e-17
std,1.000562e+00,1.000562e+00,1.000562e+00
min,-2.224156e+00,-6.484217e-01,-5.609748e-01
25%,-5.657365e-01,-4.891482e-01,-5.609748e-01
50%,-1.046374e-01,-3.573909e-01,-5.609748e-01
75%,4.333115e-01,-2.424635e-02,5.915988e-02
max,3.891554e+00,9.667167e+00,5.640372e+00


In [20]:
# Check and handle outliers (using IQR method) on Fare, since fare has extreme values
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df["Fare"] < lower) | (df["Fare"] > upper)]
print(f"Number of Fare outliers detected: {outliers.shape[0]}")
print("These are kept in the dataset since high-fare passengers are a real, meaningful group "
      "(e.g. first-class passengers), not data errors — but scaling above reduces their influence.")

Number of Fare outliers detected: 116
These are kept in the dataset since high-fare passengers are a real, meaningful group (e.g. first-class passengers), not data errors — but scaling above reduces their influence.


## 5. Dataset Preparation

In [21]:
# Separate features (X) and target (y = Survived)
X = df.drop(columns=["Survived"])
y = df["Survived"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (891, 11)
y shape: (891,)


In [22]:
# Split data into training and testing sets (80/20, random_state=42), with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("\nTrain target distribution:\n", y_train.value_counts(normalize=True))
print("\nTest target distribution:\n", y_test.value_counts(normalize=True))

X_train: (712, 11) X_test: (179, 11)

Train target distribution:
 Survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64

Test target distribution:
 Survived
0    0.614525
1    0.385475
Name: proportion, dtype: float64


## 6. Preprocessing Pipeline (Scikit-learn)

The pipeline below is fit **only on X_train** to avoid data leakage, and can
then be reused to transform X_test or any new/unseen data consistently.

In [23]:
numeric_features = ["Age", "Fare", "FamilySize", "SibSp", "Parch"]
categorical_features = ["Sex", "Embarked", "Title", "Pclass"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

full_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
])

# Fit on training data only, then transform both train and test
X_train_processed = full_pipeline.fit_transform(X_train[numeric_features + categorical_features])
X_test_processed = full_pipeline.transform(X_test[numeric_features + categorical_features])

print("Processed X_train shape:", X_train_processed.shape)
print("Processed X_test shape:", X_test_processed.shape)

Processed X_train shape: (712, 18)
Processed X_test shape: (179, 18)


## 7. Validation

In [24]:
# No missing values remain
print("Missing values in final cleaned dataset:")
print(df.isnull().sum().sum(), "total missing values")

Missing values in final cleaned dataset:
0 total missing values


In [25]:
# All features are numerical (after encoding)
non_numeric = df_encoded.drop(columns=["Survived"]).select_dtypes(exclude=[np.number, bool]).columns
print("Non-numeric columns remaining (should be empty):", list(non_numeric))

Non-numeric columns remaining (should be empty): []


In [26]:
# No data leakage: pipeline was fit only on X_train, not on the full dataset or X_test
print("Pipeline fitted only on X_train -> no data leakage into X_test.")

Pipeline fitted only on X_train -> no data leakage into X_test.


In [27]:
# Class distribution preserved (stratified split)
print("Original distribution:\n", y.value_counts(normalize=True))
print("\nTrain distribution:\n", y_train.value_counts(normalize=True))
print("\nTest distribution:\n", y_test.value_counts(normalize=True))

Original distribution:
 Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

Train distribution:
 Survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64

Test distribution:
 Survived
0    0.614525
1    0.385475
Name: proportion, dtype: float64


## 8. Save Outputs

In [28]:
# Save clean dataset
df_encoded.to_csv("titanic_clean.csv", index=False)

# Save train/test splits
X_train.to_csv("X_train.csv", index=False)
X_test.to_csv("X_test.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

# Save the preprocessing pipeline
joblib.dump(full_pipeline, "preprocessing_pipeline.pkl")

print("Saved files:")
print("- titanic_clean.csv")
print("- X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("- preprocessing_pipeline.pkl")

Saved files:
- titanic_clean.csv
- X_train.csv, X_test.csv, y_train.csv, y_test.csv
- preprocessing_pipeline.pkl
